In [1]:
from material import Material
from geometry import Geometry
from srim_output import SRIMOutput



# ============================================================
# SRIM OUTPUT:
# 5.3 MeV He in NbN
# ============================================================

he_nbn_srim = SRIMOutput(

    # ion properties
    ion_name="He",
    ion_energy_eV=5.3e6,
    ion_mass_amu=4.003,

    # stopping powers
    #
    # SRIM units:
    #   keV / micron
    #
    # Convert:
    #   1 keV/micron =
    #   1e3 * 1.602e-19 / 1e-6
    #   = 1.602e-10 J/m
    #

    dEdx_electronic=(
        3.510e2 * 1.602e-10
    ),

    dEdx_nuclear=(
        2.727e-1 * 1.602e-10
    ),

    # projected range
    #
    # 10.18 micron → meters
    #
    range=10.18e-6,

    # longitudinal straggling
    #
    # 4503 A → meters
    #
    straggle_long=4503e-10,

    # lateral straggling
    #
    # 5510 A → meters
    #
    straggle_trans=5510e-10,
)
print(he_nbn_srim.summary())

{'ion': 'He', 'E_eV': 5300000.0, 'range_m': 1.018e-05, 'dEdx_electronic': 5.62302e-08, 'dEdx_nuclear': 4.368654e-11, 'dEdx_total': 5.6273886539999996e-08, 'electronic_fraction': 0.9992236800639503, 'nuclear_fraction': 0.0007763199360496845}


In [3]:


# ============================================================
# DEFINE MATERIAL
# ============================================================

nbn = Material(

    name="NbN",

    density=8470.0,               # kg/m^3
    cv=2e6,                       # J/m^3/K
    thermal_conductivity=10.0,    # W/m/K

    Tc=9.0,
    g_ep=1e17, #???? lol!

    srim=he_nbn_srim
)

print(nbn)

Material(name='NbN', density=8470.0, cv=2000000.0, thermal_conductivity=10.0, Tc=9.0, g_ep=1e+17, srim=SRIMOutput(ion_name='He', ion_energy_eV=5300000.0, ion_mass_amu=4.003, dEdx_electronic=5.62302e-08, dEdx_nuclear=4.368654e-11, range=1.018e-05, straggle_long=4.503e-07, straggle_trans=5.51e-07), stopping_power_scale=1.0, damage_efficiency=1.0)


In [5]:
# ============================================================
# DEFINE SAMPLE GEOMETRY
# ============================================================

geom = Geometry(

    # sample center
    location=(0.0, 0.0, 0.0),

    # dimensions
    lx=100e-6, #x is direction of current flow, pick commensurate with width for now as width >> thickness 
    ly=100e-6, # width

    # thin NbN film
    lz=200e-9, #200 nm

    material=nbn
)

print(geom)

Geometry(location=array([0., 0., 0.]), lx=0.0001, ly=0.0001, lz=2e-07, material=Material(name='NbN', density=8470.0, cv=2000000.0, thermal_conductivity=10.0, Tc=9.0, g_ep=1e+17, srim=SRIMOutput(ion_name='He', ion_energy_eV=5300000.0, ion_mass_amu=4.003, dEdx_electronic=5.62302e-08, dEdx_nuclear=4.368654e-11, range=1.018e-05, straggle_long=4.503e-07, straggle_trans=5.51e-07), stopping_power_scale=1.0, damage_efficiency=1.0))


In [6]:

# ============================================================
# EXAMPLE USAGE
# ============================================================

print("Material:", geom.material.name)

print("Film thickness (m):", geom.thickness())

print("Sample volume (m^3):", geom.volume())

print("Thermal diffusivity:")
print(geom.material.thermal_diffusivity())

print("Total stopping power (J/m):")
print(geom.material.dEdx_total)

print("Electronic stopping fraction:")
print(geom.material.electronic_fraction)

print("Nuclear stopping fraction:")
print(geom.material.nuclear_fraction)


# ============================================================
# CREATE GRID
# ============================================================

X, Y, Z = geom.create_grid(
    nx=50,
    ny=50,
    nz=10
)

print("Grid shape:", X.shape)


# ============================================================
# POINT CONTAINMENT TEST
# ============================================================

p = [0.0, 0.0, 0.0]

print("Point inside sample?")
print(geom.contains(p))

Material: NbN
Film thickness (m): 2e-07
Sample volume (m^3): 1.9999999999999998e-15
Thermal diffusivity:
5e-06
Total stopping power (J/m):
5.6273886539999996e-08
Electronic stopping fraction:
0.9992236800639503
Nuclear stopping fraction:
0.0007763199360496845
Grid shape: (50, 50, 10)
Point inside sample?
True
